In [55]:
import hashlib
import json
import os
from dataclasses import dataclass, field
from typing import Any
from urllib.parse import urljoin

import pandas as pd
from parsel import Selector

from longscrape import InMemoryTaskQueue, RichEntry, Task
from longscrape.adapters.playwright.fetcher import DefaultFetcher
from longscrape.adapters.playwright.middlewares import URLBlocklist, URLCacher
from longscrape.adapters.playwright.patchright import PatchrightManager
from longscrape.adapters.store.raw_entry import PyMongoRawEntryStore

In [56]:
@dataclass
class Company:
    """One company found in an Aleo search card or detail page."""

    id: str
    name: str | None
    details_url: str
    source: str
    nip: str | None = None
    krs: str | None = None
    regon: str | None = None
    address: dict[str, Any] | None = None
    contact: dict[str, str | None] = field(default_factory=dict)
    categories: list[list[str]] = field(default_factory=list)
    registry_data: dict[str, Any] = field(default_factory=dict)
    financial_data: dict[str, Any] = field(default_factory=dict)
    person_ids: list[str] = field(default_factory=list)


@dataclass
class Person:
    """A person associated with a detailed Company entry."""

    id: str
    name: str
    company_id: str
    age: int | None = None
    roles: list[dict[str, Any]] = field(default_factory=list)

In [57]:
# URLCacher persists browser responses in playbooks/.cache; Mongo stores final HTML by task cache key.
playwright = PatchrightManager(headless=False)
playwright.register_middleware(URLBlocklist())
playwright.register_middleware(URLCacher())
await playwright.start()

fetcher = DefaultFetcher(playwright, "aleo.com")
raw_entries = PyMongoRawEntryStore(
    os.environ.get("MONGODB_URI", "mongodb://localhost:27017")
)
await raw_entries.start()
queue = InMemoryTaskQueue()

In [58]:
ALEO_TASK_KIND = "aleo"
ALEO_DETAILS_TASK_KIND = "aleo.details"
ALEO_ORIGIN = "https://aleo.com/int/"
SEARCH_URL = "https://aleo.com/int/companies/it-i-telekomunikacja/doradztwo-techniczne?voivodeships=LODZ&city=%C5%81%C3%B3d%C5%BA"

task = Task(kind=ALEO_TASK_KIND, query=SEARCH_URL)

In [59]:
raw_entry = await raw_entries.get(task.cache_key)
if raw_entry is None:
    raw_entry = await fetcher.fetch(task)
    await raw_entries.put(task.cache_key, raw_entry)

In [60]:
def clean_text(values: list[str]) -> str | None:
    value = " ".join(part.strip() for part in values if part.strip())
    return value or None


def normalize_name(first_name: str | None, last_name: str | None) -> str:
    return " ".join(part for part in (first_name, last_name) if part).strip()


def find_api_response(state: dict[str, Any], suffix: str) -> dict[str, Any]:
    for response in state.values():
        if isinstance(response, dict) and suffix in response.get("u", ""):
            return response.get("b", {})
    return {}


def extract_search_companies(html: str) -> list[Company]:
    page = Selector(text=html)
    companies = []
    for card in page.css("app-base-catalog-row"):
        link = card.css("a.catalog-row-first-line__company-name")
        href = link.attrib.get("href")
        if not href:
            continue
        nip = clean_text(card.css(".tax-id::text").getall())
        krs = clean_text(card.css(".krs::text").getall())
        regon = clean_text(card.css(".regon::text").getall())
        company_id = krs or regon or nip or href.rsplit("/", 1)[-1]
        companies.append(
            Company(
                id=company_id,
                name=clean_text(link.css("::text").getall()),
                details_url=urljoin(ALEO_ORIGIN, href),
                source="search",
                nip=nip,
                krs=krs,
                regon=regon,
                address={
                    "display": clean_text(
                        card.css(".catalog-row-company-info__address ::text").getall()
                    )
                },
                categories=[
                    [
                        value.strip()
                        for value in card.css(
                            ".category-strap .category-label::text"
                        ).getall()
                        if value.strip()
                    ]
                ],
                registry_data={
                    "rating": clean_text(
                        card.css(".catalog-row-rating ::text").getall()
                    ),
                    "description": clean_text(
                        card.css("app-company-catalog-row-bottom p::text").getall()
                    ),
                },
            )
        )
    return companies


def extract_detail_entries(
    html: str, url: str
) -> tuple[RichEntry[Company], list[RichEntry[Person]]]:
    page = Selector(text=html)
    state_json = page.css("script#ng-state::text").get()
    if not state_json:
        raise ValueError("Aleo detail page has no ng-state payload")
    state = json.loads(state_json)
    company_data = find_api_response(state, "/api/v2/companies")
    registry_details = find_api_response(state, "krs-registry-data")
    relations = find_api_response(state, "/relations")
    financial_data = {
        key: registry_details.get(key)
        for key in ("rankings", "financialIndicators", "financialDocuments")
    }

    identification = company_data.get("identification", {})
    registry_data = company_data.get("registryData", {})
    # KRS is stable across the lightweight search entry and the enriched detail entry.
    company_id = (
        identification.get("krs")
        or identification.get("regon")
        or identification.get("nip")
        or identification.get("id")
    )
    if not company_id:
        raise ValueError("Aleo detail page has no stable company identifier")

    categories = [
        [item.get("name") for item in category.get("paths", []) if item.get("name")]
        for category in company_data.get("categories", [])
    ]
    people_by_name = {}
    for node in relations.get("personalNodes", []):
        name = node.get("name")
        if name:
            relation_roles = [
                {"type": label.get("connectionType"), "label": label.get("label")}
                for relation in node.get("data", [])
                for label in relation.get("labels", [])
                if relation.get("companyId") == identification.get("id")
            ]
            people_by_name[name] = {
                "id": str(node["id"]),
                "age": node.get("age"),
                "roles": relation_roles,
            }

    authority_groups = registry_details.get("authorities", {})
    for group, authorities in authority_groups.items():
        if not isinstance(authorities, list):
            continue
        for authority in authorities:
            name = normalize_name(authority.get("firstName"), authority.get("lastName"))
            if not name:
                continue
            person = people_by_name.setdefault(
                name,
                {
                    "id": hashlib.sha256(f"{company_id}:{name}".encode()).hexdigest(),
                    "age": authority.get("age"),
                    "roles": [],
                },
            )
            person["roles"].append(
                {
                    "type": group,
                    "function": authority.get("function"),
                    "shares": authority.get("shares"),
                    "suspended": authority.get("isSuspended"),
                }
            )

    people = [
        Person(
            id=value["id"],
            name=name,
            company_id=company_id,
            age=value["age"],
            roles=value["roles"],
        )
        for name, value in people_by_name.items()
    ]
    company = Company(
        id=company_id,
        name=registry_data.get("name")
        or clean_text(page.css(".text-company-name::text").getall()),
        details_url=url,
        source="details",
        nip=identification.get("nip"),
        krs=identification.get("krs"),
        regon=identification.get("regon"),
        address=(company_data.get("addresses") or [{}])[0],
        contact={
            key: registry_data.get(key) for key in ("siteUrl", "phoneNumber", "email")
        },
        categories=categories,
        registry_data={
            **registry_data,
            "shareCapital": company_data.get("shareCapital"),
            "bankAccounts": company_data.get("bankAccounts"),
            "raw_registry": registry_details,
        },
        financial_data=financial_data,
        person_ids=[person.id for person in people],
    )
    return RichEntry(url=url, data=company), [
        RichEntry(url=url, data=person) for person in people
    ]

In [61]:
search_companies = extract_search_companies(raw_entry.content)
search_entries = [
    RichEntry(url=company.details_url, data=company) for company in search_companies
]
detail_tasks = [
    Task(kind=ALEO_DETAILS_TASK_KIND, query=company.details_url)
    for company in search_companies
]
for detail_task in detail_tasks:
    await queue.put(detail_task)

# Lightweight search-stage companies intentionally have no people associations.
search_table = pd.DataFrame(
    [
        {
            "company_id": company.id,
            "name": company.name,
            "nip": company.nip,
            "krs": company.krs,
            "regon": company.regon,
            "address": (company.address or {}).get("display"),
            "categories": " / ".join(company.categories[0])
            if company.categories
            else None,
            "detail_url": company.details_url,
        }
        for company in search_companies
    ]
)
search_table.head(10)

,company_id,name,nip,krs,regon,address,categories,detail_url
0,0000669769,HUBNERBIT SPÓŁKA Z OGRANICZONĄ ODPOWIEDZIALNOŚCIĄ,7272812046,0000669769,366851409,"ŁÓDŹ, ŁÓDŹ",IT i telekomunikacja / Doradztwo techniczne,https://aleo.com/int/company/hubnerbit-spolka-...
1,0000794774,ASYSTENT NAUCZYCIELA SPÓŁKA Z OGRANICZONĄ ODPO...,7252291874,0000794774,384063649,"UL. PREZYDENTA GABRIELA NARUTOWICZA 40 / 1, 90...",IT i telekomunikacja / Doradztwo techniczne,https://aleo.com/int/company/asystent-nauczyci...
2,0000801298,WUNDERWOLKEN SPÓŁKA Z OGRANICZONĄ ODPOWIEDZIAL...,7252292253,0000801298,384235790,"UL. PRZĘDZALNIANA 127 / 66, 93-286 ŁÓDŹ",IT i telekomunikacja / Doradztwo techniczne,https://aleo.com/int/company/wunderwolken-sp-z...
3,0000547964,NOWOCZESNE ROZWIĄZANIA INFORMATYCZNE SPÓŁKA Z ...,7262656113,0000547964,360997425,"UL. PREZYDENTA GABRIELA NARUTOWICZA 40/1, 90-1...",IT i telekomunikacja / Doradztwo techniczne,https://aleo.com/int/company/nowoczesne-rozwia...
4,0000821465,BESTPROJECTS SPÓŁKA Z OGRANICZONĄ ODPOWIEDZIAL...,7282842534,0000821465,385208110,"UL. PREZYDENTA GABRIELA NARUTOWICZA 40 / 1, 90...",IT i telekomunikacja / Doradztwo techniczne,https://aleo.com/int/company/bestprojects-spol...
5,0000961850,SADDLEFIT SPÓŁKA Z OGRANICZONĄ ODPOWIEDZIALNOŚCIĄ,9472007184,0000961850,521529510,"UL. BRUKOWA 12, 91-341 ŁÓDŹ",IT i telekomunikacja / Doradztwo techniczne,https://aleo.com/int/company/saddlefit-spolka-...
6,0000420422,"""AMB ENERGY"" SPÓŁKA Z OGRANICZONĄ ODPOWIEDZIAL...",7292701723,0000420422,101408065,"UL. SREBRZYŃSKA 53 / 30, 91-087 ŁÓDŹ",IT i telekomunikacja / Doradztwo techniczne,https://aleo.com/int/company/amb-energy-spolka...
7,0000755160,2XJ SPÓŁKA Z OGRANICZONĄ ODPOWIEDZIALNOŚCIĄ W ...,9820381365,0000755160,381676825,"UL. GABRIELI ZAPOLSKIEJ 101, 93-256 ŁÓDŹ",IT i telekomunikacja / Doradztwo techniczne,https://aleo.com/int/company/2xj-sp-z-oo-lodz
8,0000708176,"""ALTICA"" SPÓŁKA Z OGRANICZONĄ ODPOWIEDZIALNOŚCIĄ",7292721157,0000708176,368941855,"UL. STANISŁAWA DUBOIS 114 / 116, 93-465 ŁÓDŹ",IT i telekomunikacja / Doradztwo techniczne,https://aleo.com/int/company/altica-sp-z-oo-lodz
9,0001044933,ADDICTIVE SPÓŁKA Z OGRANICZONĄ ODPOWIEDZIALNOŚCIĄ,7282874586,0001044933,525721871,"AL. MARSZ. JÓZEFA PIŁSUDSKIEGO 92, 92-202 ŁÓDŹ",IT i telekomunikacja / Doradztwo techniczne,https://aleo.com/int/company/addictive-spolka-...


In [62]:
detail_task = detail_tasks[0]
detail_raw_entry = await raw_entries.get(detail_task.cache_key)
if detail_raw_entry is None:
    detail_raw_entry = await fetcher.fetch(detail_task)
    await raw_entries.put(detail_task.cache_key, detail_raw_entry)

In [63]:
company_entry, people_entries = extract_detail_entries(
    detail_raw_entry.content, detail_raw_entry.url
)

# The Company and each Person are linked through the stable KRS company_id.
company_table = pd.DataFrame(
    [
        {
            "company_id": company_entry.data.id,
            "name": company_entry.data.name,
            "nip": company_entry.data.nip,
            "krs": company_entry.data.krs,
            "regon": company_entry.data.regon,
            "address": (company_entry.data.address or {}).get("address"),
            "city": (company_entry.data.address or {}).get("city"),
            "people": len(company_entry.data.person_ids),
            "categories": len(company_entry.data.categories),
            "financial_indicators": len(
                company_entry.data.financial_data.get("financialIndicators") or []
            ),
        }
    ]
)
people_table = pd.DataFrame(
    [
        {
            "person_id": person.data.id,
            "name": person.data.name,
            "company_id": person.data.company_id,
            "age": person.data.age,
            "roles": "; ".join(
                sorted(
                    {role.get("type") for role in person.data.roles if role.get("type")}
                )
            ),
            "ownership": "; ".join(
                role["shares"] for role in person.data.roles if role.get("shares")
            ),
        }
        for person in people_entries
    ]
)
company_table, people_table

(   company_id                                               name         nip  \
 0  0000669769  HUBNERBIT SPÓŁKA Z OGRANICZONĄ ODPOWIEDZIALNOŚCIĄ  7272812046   
 
           krs      regon address  city  people  categories  \
 0  0000669769  366851409    ŁÓDŹ  ŁÓDŹ       2          10   
 
    financial_indicators  
 0                    15  ,
      person_id                         name  company_id  age  \
 0   -327928928         KAJETAN KAMIL HÜBNER  0000669769   34   
 1  -2107509970  KATARZYNA ZOFIA PIERZGALSKA  0000669769   34   
 
                                                roles  \
 0  BOARD_MEMBER; SHAREHOLDER; boardMembers; share...   
 1  BOARD_MEMBER; SHAREHOLDER; boardMembers; share...   
 
                                          ownership  
 0  9 UDZIAŁÓW O ŁĄCZNEJ WYSOKOŚCI 12915,00 ZŁOTYCH  
 1              1 UDZIAŁ O WARTOŚCI 1435,00 ZŁOTYCH  )

In [64]:
await playwright.stop()